# ML-08 — Capstone Modeling Lane

[%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib]

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook builds, evaluates, and interprets models for our **Low CTR on Page 1** refresh opportunity lane. We compare three machine learning models of varying complexity against our Week 4 rule-based baseline under an honest client-holdout validation design.

## 1. Method choice and why

### Chosen Methods and Rationale
To evaluate our lane's targets, we train three models representing different tradeoffs of complexity vs. interpretability:
1. **Logistic Regression (with StandardScaler)**: A linear baseline that provides readable coefficients, showing the directional impact of features (e.g., does higher average position or lower CTR increase decline risk?).
2. **Decision Tree (max_depth=5, min_samples_leaf=50)**: A non-linear model that extracts human-readable decision rules. It allows content managers to see exactly what combinations of thresholds (e.g., impressions and CTR) predict decline.
3. **Random Forest (n_estimators=200, max_depth=10, min_samples_leaf=25)**: An ensemble method designed to capture complex feature interactions, representing the predictive ceiling of our feature set.

Each model outputs a probability of search decline (`is_declining_label == 1`), which we can sort to rank the refresh queue.

In [1]:
import pandas as pd
import numpy as np

# Load processed dataset
df = pd.read_csv("../../data/processed/refresh_feature_vector.csv")
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")

Loaded dataset with 30000 rows and 52 columns.


## 2. Split design

### Client-Holdout Group Split
Standard randomized train/test splits suffer from **client-level leakage**. Multiple pages from the same client share domain-wide traits (such as site authority, layout, and niche). If we train on some pages of a client and test on others, the model can 'memorize' client characteristics rather than learning general signals of content decay.

To prevent this, we group the dataset by `client_id` and hold out **20% of clients** (6 clients out of 32) entirely as our test set. The models are trained on the remaining 26 clients. The baseline and the models are evaluated on the exact same test clients to guarantee an honest comparison.

In [2]:
# Feature list definitions
MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

# Preprocess feature matrix
numeric_frame = df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
categorical_frame = df[MODEL_CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=MODEL_CATEGORICAL_FEATURES, dummy_na=False, dtype=float)

X = pd.concat([numeric_frame, encoded_frame], axis=1)
y = df["is_declining_label"]
clients = df["client_id"].fillna("unknown").astype(str)

# Group split by client_id
unique_clients = clients.drop_duplicates().to_numpy()
np.random.seed(42)
shuffled_clients = np.random.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = clients.isin(test_clients).to_numpy()

train_indices = np.arange(len(df))[~test_mask]
test_indices = np.arange(len(df))[test_mask]

X_train, y_train = X.iloc[train_indices], y.iloc[train_indices]
X_test, y_test = X.iloc[test_indices], y.iloc[test_indices]
df_test = df.iloc[test_indices].copy()

print(f"Total clients: {len(unique_clients)}")
print(f"Test clients ({len(test_clients)}): {test_clients}")
print(f"Train size: {X_train.shape[0]} rows, Test size: {X_test.shape[0]} rows")
print(f"Train base rate: {y_train.mean():.3f}, Test base rate: {y_test.mean():.3f}")
assert len(set(df.iloc[train_indices]['client_id']).intersection(test_clients)) == 0, "Client leakage detected!"

Total clients: 32
Test clients (6): {'client_8b940be7fb', 'client_a88a7902cb', 'client_8527a891e2', 'client_9f14025af0', 'client_9400f1b21c', 'client_bbb965ab0c'}
Train size: 26619 rows, Test size: 3381 rows
Train base rate: 0.544, Test base rate: 0.525


## 3. Train + compare vs my baseline

### Training and Evaluation Comparison
We fit all models on the training partition and generate decline probability scores on the test partition. 
To evaluate opportunity-ranking capability, we compute **Precision@10** and **Precision@50** on the test set alongside global metrics (**ROC AUC** and **Average Precision**).

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Define training pipelines
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])
dt = DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=42)
rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)

# Fit models
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

# Calculate Baseline Action Score on the Test Set
is_page_1 = (df_test["avg_position"] > 0) & (df_test["avg_position"] <= 10)
is_visible = df_test["impressions_90d"] >= 500
is_low_ctr = df_test["ctr"] < 0.20  # below-average CTR for page 1
matches_rule = is_page_1 & is_visible & is_low_ctr

visibility_score = df_test["impressions_90d"].rank(pct=True)
ctr_min = df_test.loc[matches_rule, "ctr"].min() if matches_rule.any() else 0
ctr_max = df_test.loc[matches_rule, "ctr"].max() if matches_rule.any() else 1
ctr_norm = (df_test["ctr"] - ctr_min) / (ctr_max - ctr_min + 1e-6)
ctr_risk_score = 1.0 - ctr_norm.clip(0, 1)

df_test["baseline_score"] = np.where(matches_rule, 0.5 * visibility_score + 0.5 * ctr_risk_score, 0.0)

# Evaluate predictions
def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

models = {
    "Baseline Rule": df_test["baseline_score"],
    "Logistic Regression": lr.predict_proba(X_test)[:, 1],
    "Decision Tree": dt.predict_proba(X_test)[:, 1],
    "Random Forest": rf.predict_proba(X_test)[:, 1],
}

print("=== MODEL COMPARISON ON TEST SET ===")
print(f"{'Model':<25} | {'P@10':<6} | {'P@50':<6} | {'ROC AUC':<8} | {'AP':<6}")
print("-" * 60)
for name, probs in models.items():
    p10 = precision_at_k(y_test, probs, 10)
    p50 = precision_at_k(y_test, probs, 50)
    auc = roc_auc_score(y_test, probs)
    ap = average_precision_score(y_test, probs)
    print(f"{name:<25} | {p10:<6.3f} | {p50:<6.3f} | {auc:<8.3f} | {ap:<6.3f}")

=== MODEL COMPARISON ON TEST SET ===
Model                     | P@10   | P@50   | ROC AUC  | AP    
------------------------------------------------------------
Baseline Rule             | 0.900  | 0.860  | 0.518    | 0.542 
Logistic Regression       | 0.800  | 0.760  | 0.652    | 0.642 
Decision Tree             | 0.800  | 0.740  | 0.664    | 0.637 
Random Forest             | 0.400  | 0.420  | 0.659    | 0.622 


## 4. Errors and interpretation

### Feature Importance and Error Diagnostics
1. **Feature Importances**: The Random Forest model relies heavily on historical traffic visibility and page age (`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`).
2. **Overfitting and Domain Shift**: Although Random Forest has a reasonable global ROC AUC (0.659), its Precision@10 is extremely poor (**0.400**), performing worse than a random guess (base rate: 0.525). Random Forest overfit to complex interactions that relate to specific clients in the training set. When evaluated on the held-out test clients (whose authority/domain patterns differ), it recommended pages that were actually stable (`is_declining_label == 0`).
3. **Generalization of Simple Rules**: In contrast, the **Baseline Rule** achieved a Precision@10 of **0.900** and Precision@50 of **0.860**, while **Logistic Regression** achieved **0.800** and **0.760**. Simple linear or heuristic models generalize much better because they do not overfit to complex site-specific feature interactions.

In [4]:
# Print top features for Random Forest
importances = rf.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
print("=== TOP 10 FEATURES (Random Forest) ===")
print(feat_imp.head(10))

# Analyze top 3 False Positives (predicted high risk but stable)
df_test["rf_prob"] = models["Random Forest"]
fps = df_test[df_test["is_declining_label"] == 0].sort_values("rf_prob", ascending=False).head(3)
print("\n=== TOP 3 FALSE POSITIVES ===")
for i, row in fps.iterrows():
    print(f"Content ID: {row['content_id']}, client: {row['client_id']}, probability: {row['rf_prob']:.3f}")
    print(f"  Impressions: {row['impressions_90d']}, CTR: {row['ctr']:.4f}, Position: {row['avg_position']}, Days since update: {row['days_since_last_update']}")

=== TOP 10 FEATURES (Random Forest) ===
days_with_impressions    0.138839
log_impressions_90d      0.119145
avg_position             0.106483
content_age_days         0.095810
word_count               0.050823
char_count               0.041193
ctr                      0.034552
log_clicks_90d           0.033669
scroll_rate              0.029025
days_with_sessions       0.029018
dtype: float64

=== TOP 3 FALSE POSITIVES ===
Content ID: content_2ba626fea4d6, client: client_8527a891e2, probability: 0.852
  Impressions: 360, CTR: 0.0000, Position: 7.2, Days since update: 104
Content ID: content_a5a2fbc76336, client: client_8527a891e2, probability: 0.836
  Impressions: 307, CTR: 0.0000, Position: 39.8, Days since update: 103
Content ID: content_8f1409b2674e, client: client_8527a891e2, probability: 0.832
  Impressions: 209, CTR: 0.0000, Position: 20.0, Days since update: 104


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.